# Tools

In [1]:
from httpx._transports import default
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from pathlib import Path

from langchain_core.messages import HumanMessage, ToolMessage
from rich import print as rprint
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 从.env文件中加载环境变量
load_dotenv(Path('../.env'), override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model='qwen-max',
    model_provider='openai',
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
)


## 工具调用-ChatPromptTemplate封装提示词

In [3]:
from langchain_core.tools import tool


@tool(description="获取城市天气")
def get_weather(city: str):
    return f'{city}今天晴天，温度28度'


# rprint(type(get_weather))
result = get_weather.invoke({'city': '北京'})
print(result)

chat_model = model.bind_tools([get_weather])

config = {
    "run_name": "天气助手",  # 在LangSmith中这次运行会显示为
}

prompt = ChatPromptTemplate(
    [
        ('system', '你是一个智能助手'),
        MessagesPlaceholder(variable_name='history')
    ]
)
history = []
user_input = '深圳今天天气怎么样?'
history.append(('user', user_input))
prompt_value = prompt.invoke(
    {
        'history': history
    }
)

# invoke是langchain封窗的可调用对象
resp = chat_model.invoke(prompt_value, config=config)
history.append(resp)

rprint(resp)
print('-' * 40)

if resp.tool_calls:
    for call in resp.tool_calls:
        if call['name'] == 'get_weather':
            tool_result = get_weather.invoke(call['args'])
            tool_message = ToolMessage(
                content=tool_result,
                tool_call_id=call['id'],
                name=call['name']
            )
            history.append(tool_message)
    prompt_value = prompt.invoke({'history': history})
    final_resp = chat_model.invoke(prompt_value, config=config)
    history.append(final_resp)
    rprint(final_resp.content)
else:
    rprint(resp.content_blocks)



<class 'langchain_core.tools.structured.StructuredTool'>

## 工具调用-普通列表封装提示词

In [ ]:
# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])

# 声明一个消息列表
messages = [HumanMessage("今天北京天气如何")]

# 模型生成调用工具请求
response = model_with_tools.invoke(messages)

# 添加AIMessage到消息列表中
messages.append(response)

# rprint(response)

tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        # 大模型和Agent的主要区别在于：大模型不会主动的调用工具，所以这时候我们需要主动让工具调用。
        # 返回的是ToolMessage类型消息，添加到消息列表中
        tool_response = get_weather.invoke(tool_call)
        print(type(tool_response))
        messages.append(tool_response)

print("=====================> messages <=====================")
for msg in messages:
    msg.pretty_print()
print("=====================> messages <=====================")
final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

<class 'langchain_core.messages.tool.ToolMessage'>
=====================> messages <=====================
================================ Human Message =================================

今天北京天气如何
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_7ac34297626e4e8882a79e)
 Call ID: call_7ac34297626e4e8882a79e
  Args:
    city: 北京
================================= Tool Message =================================
Name: get_weather

北京今天晴天，温度28度
=====================> messages <=====================
final_response: 
content='北京今天是晴天，温度是28度。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 265, 'total_tokens': 280, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-max', 'system_fingerprint': None, 'id': 'chatcmpl-6c958461-c1a8-9c43-9896-8a

## 工具声明方式
### @tool或者docstring基本使用
1. 在`model.bind_tools()`中，会调用`convert_to_openai_tool()`转为tools，即便这个function没有被`@tool`装饰器。但在开发过程中还是要加上`@tool`，这样才会校验func是否有`docstring`来描述工具，能提高工具的识别度，工具调用更加稳定。如果加了`@tool`但是没有写`docstring`，会报：`ValueError: Function must have a docstring if description not provided.`，但是如果写了`@tool`，会把整个`docstring`当做`description`，需要显式指定解析docstring,`@tool(parse_docstring=True)`
2. `@tool(parse_docstring=True)`不会读取参数列表，而是 通过`typing.get_type_hints()`获取`annotations: dict[参数名： 参数类型]`, 如果`docstring_arg not in annotations`就会抛`ValueError`,所以工具func的参数**必须**加类型注解
3. `docstring`建议遵循Google Style, 快捷设置: pycharm -> settings -> Python -> Tools -> Integrated Tools -> docstring format
    - 错误示例:
        -  ```python
            """
                获取实时天气信息
                :param city: 具体的城市名
                :return: 天气详情
            """
            ```
    - 正确示例:
        -  ```python
                 """
                    获取实时天气信息

                    Args:
                        city : 具体的城市

                    Returns:
                        返回城市的天气
                """
           ```
4`@tool`的`description`优先级高于`docstring`
5`@tool`的`name_or_callable`指定tool名称，默认为func的名字，不要使用`config`和`runtime`，这是LangChain内部保留的。

In [31]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool


@tool(parse_docstring=True, name_or_callable='getWeather')
def get_weather(city: str):
    """
    获取实时天气信息

    Args:
        city : 具体的城市

    Returns:
        返回城市的天气
    """
    return f'{city}今天晴天，温度28摄氏度'


def get_stock_info(stock_num: str):
    return f"{stock_num}股票上涨"


tool_schema = convert_to_openai_tool(get_weather)
rprint(tool_schema)

{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '获取实时天气信息',
        'parameters': {
            'properties': {'city': {'description': '具体的城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 工具声明方式：args_schema
### 方式一: Pydantic模型定义
- 模型类要继承`BaseModel`
- 在字段要求枚举、范围限定的复杂场景下，Pydantic的`Field`可以非常好的处理
- args_schema的Args描述优先级**高于**docstring

In [36]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.utils.function_calling import convert_to_openai_tool


class WeatherInput(BaseModel):
    city: str
    unit: Literal["c", 'f'] = Field(
        default="c",
        description="温度(国际)单位"
    )
    meta_data: str = Field(
        description="元数据",
        default="",
        max_length=1024,
        min_length=0,
    )


@tool(args_schema=WeatherInput, parse_docstring=True, description="获取实时天气信息")
def get_weather(city: str, unit: Literal['c', 'f'], meta_data: str):
    """
    实时获取天气

    Args:
        city: 具体的城市
        unit: 温度单位
        meta_data: 元数据111

    Returns:
        具体的天气信息
    """
    if not meta_data:
        print(meta_data)
    return f'{city}今天晴天，28 {unit}度'


tool_schema = convert_to_openai_tool(get_weather)
rprint(tool_schema)

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取实时天气信息',
        'parameters': {
            'properties': {
                'city': {'type': 'string'},
                'unit': {'default': 'c', 'description': '温度(国际)单位', 'enum': ['c', 'f'], 'type': 'string'},
                'meta_data': {
                    'default': '',
                    'description': '元数据',
                    'maxLength': 1024,
                    'minLength': 0,
                    'type': 'string'
                }
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

### 方式二： JSON Schema
- JSON Schema能够在运行时动态读取配置加载，适合灵活多变的情况
- **结构**：只需要`tool_schema.function.parameters`
- 优先级**高于**docstring

In [39]:
from typing import Literal
from langchain_core.utils.function_calling import convert_to_openai_tool

tool_json_schema = {
    'city': {'type': 'string'},
    'unit': {'default': 'c', 'description': '温度(国际)单位', 'enum': ['c', 'f'], 'type': 'string'},
    'meta_data': {
        'default': '',
        'description': '元数据',
        'maxLength': 1024,
        'minLength': 0,
        'type': 'string'
    }
}


@tool(args_schema=tool_json_schema, parse_docstring=True, description="获取实时天气信息")
def get_weather(city: str, unit: Literal['c', 'f'], meta_data: str):
    """
    实时获取天气

    Args:
        city: 具体的城市
        unit: 温度单位
        meta_data: 元数据111

    Returns:
        具体的天气信息
    """
    if not meta_data:
        print(meta_data)
    return f'{city}今天晴天，28 {unit}度'


tool_schema = convert_to_openai_tool(get_weather)
rprint(tool_schema)

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取实时天气信息',
        'parameters': {
            'city': {'type': 'string'},
            'unit': {'default': 'c', 'description': '温度(国际)单位', 'enum': ['c', 'f'], 'type': 'string'},
            'meta_data': {
                'default': '',
                'description': '元数据',
                'maxLength': 1024,
                'minLength': 0,
                'type': 'string'
            }
        }
    }
}

## 扩展：tool_choice参数
- tool_choice的值（LangChain会把具体的值传给API，但具体是否生效取决于模型是否支持）：
    1. none，无论如何都不调用工具
    2. auto, 模型自动决策
    3. required, 必须调用，即使问题用不到工具（有些模型不支持）
    4. tool名称，必须调用这个工具,即使问题用不到工具

In [13]:
@tool(description="根据主题查询新闻", parse_docstring=True)
def search_news(topic: str):
    """
    根据主题查询新闻

    Args:
        topic: 新闻主题

    Returns:

    """
    return f"{topic}:的新闻..."

@tool(description="查询指定公司股票价格", parse_docstring=True)
def stock_info(company: str):
    """
    查询指定公司股票价格

    Args:
        company: 公司名称

    Returns:

    """
    return f'今天{company}的股票价格是：110元'

model_with_tools = model.bind_tools([search_news, stock_info], tool_choice='stock_info')

resp = model_with_tools.invoke('今天天气怎么样？')
# resp = model_with_tools.invoke('今天腾讯股票价格怎么样？')


rprint(resp)

AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 11,
            'prompt_tokens': 305,
            'total_tokens': 316,
            'completion_tokens_details': None,
            'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 256}
        },
        'model_provider': 'openai',
        'model_name': 'qwen-max',
        'system_fingerprint': None,
        'id': 'chatcmpl-ba900fea-e845-936b-9ceb-1d4af753bde6',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--019f7e65-90ae-7a91-a35b-18c9d9b24fbf-0',
    tool_calls=[
        {
            'name': 'stock_info',
            'args': {'company': 'Apple'},
            'id': 'call_fddf59565036465ba367a4',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 305,
        'output_tokens': 11,
        'total_tokens': 316,
        'input_token_details': {'cache_read': 256},
        'output_token_details': {}
    }
)